## Visualize the output of an LCURVE MCMC run.
* (For legacy lcurve; needs updated for Python wrapper version)
* 
* Reads in a chain file, determines the number of walkers/parameters, finds a preliminary burn-in value, and plots a corner plot.
* Then updates new parameter files called *_mcmc.txt with the median posterior values (You need to update the filters list manually)
* Runs lroche for each other these *_mcmc.txt files to create lroche model light curves.
* Generates an animated visualization of the system using the "visualise" command.
* Plots a smaller corner plot that can be used as a focused corner plot figure in a manuscript.

In [ ]:
# Import libraries for loading jpg images and showing animations.
import matplotlib.image as mpimg
import matplotlib.animation as animation
from IPython.display import display, HTML
import matplotlib
matplotlib.rcParams['animation.html'] = 'jshtml'

# Standard imports
import matplotlib.pyplot as plt
import numpy as np
import time
from matplotlib import rc
rc("font",**{"family":["DejaVu Sans"]})
rc("text", usetex=False)

import scipy
import scipy.optimize
import scipy.stats as stats
from scipy.integrate import tplquad

import os
import subprocess

import seaborn as sns; sns.set(style="ticks", color_codes=True, font_scale=2.5)
import pandas as pd

In [ ]:
def get_rvol(q, r_df):
    #
    #
    #
    radii = r_df[(r_df["q"]==np.round(q,2))]
    return(radii["R_vol"].values[0])

In [ ]:
infile = "chain.mcmc"

# Build the results dataframe.
header = ["walker_id"]
with open(infile, "r") as cfile:
    for header_line_count in range(6): # Always skip the first 6 lines
        cfile.readline()
    for line in cfile.readlines(): 
        header_line_count += 1         # Count the remaining header lines with parameter values
        if len(line)==0:
            continue
        if line[0]!="#":
            break
        if len(line.split())==4:
            header.append(line.split()[1])
header.append("chi2")
df0 = pd.read_csv(infile, delimiter=" ", names=header, skiprows=header_line_count, index_col=False)

print("Fitted values:")
for i,fitted_value in enumerate(header[1:-1]):
    print(f"  {i+1:>2d}  {fitted_value}")

print()
nwalkers = max(pd.unique(df0["walker_id"])+1)
print(f"nwalkers = {nwalkers:>3d}")

ignored_lines = 0
while df0.iloc[-1]["walker_id"] != nwalkers-1:
    df0 = df0.iloc[:-1]
    ignored_lines += 1
    print(f"Incomplete walker chain detected. ignored_lines={ignored_lines}")
    if ignored_lines >= nwalkers:
        break
niters = len(df0)/nwalkers
print(f"niterations = {niters:,.0f}")

In [ ]:
nconverge = 8

fig = plt.figure(figsize=(12,5))
ax = fig.add_subplot(111)

plot_column = "chi2"
assert plot_column in df0.keys(), "plot_column variable value must match one of the dataframe header values"

iteration_count = np.arange(1, niters+1)
for k in pd.unique(df0["walker_id"]):
    sub_df0 = df0[df0["walker_id"]==k]
    ax.plot(iteration_count, sub_df0[plot_column], linewidth=0.5, alpha=0.5)

ax.grid(linestyle=":", alpha=1.0, linewidth=0.5, color="black", which="both")

ax.set_ylim(np.nanpercentile(df0[plot_column].values, 2), np.nanpercentile(df0[plot_column].values,100)/1.01)
ax.set_xlabel("Iteration #")
ax.set_ylabel(plot_column)
ax.set_title(f"N_iterations = {niters:,.0f}", loc="left")
ax.set_title(f"N_walkers = {nwalkers:,.0f}", loc="right")

# Split the resulting chi2 plot into N chunks by iteration number.
# Calculate the average chi2 value in each chunk.
# Work backwards to find when the chi2 converged.
chunk_size = 20
convergence_count = 0
burn_in = 0
avg0 = -99999
std0 = 1e-6
for i in range(chunk_size, -1, -1):
    range1 = int(i * (nwalkers*niters//chunk_size))
    range2 = int(min((i+1) * (nwalkers*niters//chunk_size), nwalkers*niters))
    
    chi2_sample = df0.iloc[range1:range2][plot_column].values

    if len(chi2_sample) < (len(df0)/chunk_size)//3: # Require the chunk to be at least 30% filled before considering converged.
        continue
    
    avg = np.nanmean(chi2_sample)
    std = np.nanstd(chi2_sample)

    if abs(avg - avg0)/std0 < 0.5:
        convergence_count += 1
        
        if convergence_count >= nconverge and burn_in==0:            
            burn_in = range2//nwalkers*nwalkers
    else:
        convergence_count = 0
        
    plt.errorbar(np.average([range1, range2])/nwalkers, avg, yerr=std, color="red", capsize=1, elinewidth=0.5, marker=".", alpha=1)

    
    avg0 = avg
    std0 = std

# Manually assign burn-in if the algorithm's value is clearly bad.
# burn_in = 350*nwalkers


ax.axvline(burn_in/nwalkers, label=f"Burn-in: {burn_in/nwalkers:,.0f}")
ax.legend(fontsize=12)
plt.show()

df = df0.iloc[burn_in:][[*header[1:-1]]].sample(15000) # Remove burn-in rows and remove columns for walker_id and chi2.
print(len(df)/nwalkers)

In [ ]:
# Calculate absolute values and add them to the dataframe.

# Extract the orbital period from a parameters file.
parameter_filename = "parameters_g.txt"
with open(parameter_filename) as pfile:
    for line in pfile.readlines():
        if line[:6]=="period":
            period = float(line.split()[2])
            break
            
print(f"Orbital period = {period} d  ({parameter_filename})")
print(f"               = {period*1440:<7.3f} min")
print(f"               = {period*86400:<7.1f} s\n")

G     = 6.67e-11   # mks
M_sun = 1.988e+30  # mks
R_sun = 6.957e8    # mks

df["M2"] = df["q"]/(1+df["q"])*period*(24*3600)/(2*np.pi*G)*(df["velocity_scale"]*1000)**3/M_sun
df["M1"] = 1./(1+df["q"])*period*(24*3600)/(2*np.pi*G)*(df["velocity_scale"]*1000)**3/M_sun
df["a"]   = period*(24*3600)/(2*np.pi)*df["velocity_scale"]*1000/R_sun
df["K2"]    = 1./(1+df["q"])*df["velocity_scale"]*np.sin(np.pi*df["iangle"]/180.)

# df["a"]   = ((period*86400)**2 * G * (2e30*(df0["M1"]+df0["M2"]))/4/np.pi/np.pi)**(1./3)/7e8


r_df = pd.read_csv("roche_radii_table.csv", comment="#")
df["R2vol"] = np.array([get_rvol(k, r_df) for k in df["q"].values]) * df["a"]

try:
    df["R1"] = df["r1"] * df["a"]
except:
    pass

df["logg2"] = np.log10(6.67e-8 * df["M2"]*(M_sun*1000) / (df["R2vol"]*(R_sun*100))**2)

df = df[(df["iangle"] > 0) & (df["temp_disc"] > 9000)]

print(df.describe())




In [ ]:
g = sns.PairGrid(df, corner=True, diag_sharey=False, despine=True, aspect=2)
g = g.map_diag(plt.hist, bins=25, histtype="step", color="black", density=True)
g = g.map_offdiag(plt.scatter, alpha=10./100, marker=".", color="black", s=1)

best_parameters = []
for i, k in enumerate(df.keys()):
    fifty     = np.percentile(df[k], 50)
    onesigup  = np.percentile(df[k], 84.13) - fifty
    onesigdwn = fifty - np.percentile(df[k], 15.87)
    print(f"{i:>2d} {k:>15s}   {{{fifty:>}}}{{{onesigup:>}}}{{{onesigdwn:>}}}")
    # print(f"{i:>2d} {k:>15s}   {fifty:>}  {onesigup:>}  {onesigdwn:>}")
    
    if k in header:
        best_parameters.append(fifty)
    
    title00 = f"{k} = ${fifty:>.5f}^{{+{onesigup:<.5f}}}_{{-{onesigdwn:<.5f}}}$"
            
    if k in ["t1", "t2", "temp_disc", "temp_spot", "temp_edge", "stsp11_tcen", "stsp12_tcen"]:
        if fifty > 1000:
            title00 = f"{k} = ${10*int(fifty/10):>5.0f}^{{+{10*int(onesigup/10):<5.0f}}}_{{-{10*int(onesigdwn/10):<5.0f}}}$"
        else:
            title00 = f"{k} = ${fifty:>6.2f}^{{+{onesigup:<6.2f}}}_{{-{onesigdwn:<6.2f}}}$"    

    elif "t0" in k:
        title00 = f"{k} = ${fifty:>5.6f}^{{+{onesigup:<.3e}}}_{{-{onesigdwn:<.3e}}}$"
    
    elif k in ["M2", "M1"]:
        title00 = f"{k} = ${fifty:>5.3f}^{{+{onesigup:<.3f}}}_{{-{onesigdwn:<.3f}}}$"

    elif k in ["r2", "r1", "R2vol", "R1vol", "R2", "R1"]:
        title00 = f"{k} = ${fifty:>5.4f}^{{+{onesigup:<.4f}}}_{{-{onesigdwn:<.4f}}}$"
        
    
    elif k in ["K2", "K1", "velocity_scale", "vsini"]:
        title00 = f"{k} = ${fifty:>5.1f}^{{+{onesigup:<.1f}}}_{{-{onesigdwn:<5.1f}}}$"
        
    else:
        title00 = f"{k} = ${fifty:>6.4f}^{{+{onesigup:>6.4f}}}_{{-{onesigdwn:>6.4f}}}$"

    g.axes[i,i].set_title(title00, loc="right")
    g.axes[i,i].axvline(x=fifty, color="crimson", linewidth=1)
    g.axes[i,i].axvline(x=fifty+onesigup, color="crimson", linestyle="--", linewidth=1)
    g.axes[i,i].axvline(x=fifty-onesigdwn, color="crimson", linestyle="--", linewidth=1)
print()
# plt.show()
g.savefig(f"full_corner_plot.jpg", dpi=150)

#

In [ ]:
# Build new parameters files to plot the resulting model.

final_fit_dict = {header[i+1]:best_parameters[i] for i in range(len(best_parameters))}

lcurve_path = "./lcurve.simg"

filters = ["B", "R", "g", "i2"]
scale_factors = {k:1 for k in filters} # Automatic scale factors found/used by lroche to scale model to data properly.

for filter in filters:
    with open(f"parameters_{filter}.txt", "r") as ifile:
        with open(f"parameters_{filter}_mcmc.txt", "w") as ofile:
            for line in ifile.readlines():
                split_line = line.split()

                if len(split_line) == 0:
                    ofile.write("\n")
                    continue

                if split_line[0] in final_fit_dict: # Checks exact names, does not handle "_filter" parameters such as slope.
                    param = split_line[0]
                    value = final_fit_dict[param] 
                    error = value/20 if param != "t0" else 30./86400
                    new_line = f"{param:<15s}= {value} {error} {error/10} 1 1\n"
                    ofile.write(new_line)
                elif split_line[0] in ["nrad",]:
                    value = 100
                    new_line = f"{split_line[0]:<15s}= {value}\n"
                    ofile.write(new_line)
                elif split_line[0] in ["nlat1f", "nlat2f", "nlat1c", "nlat2c"]:
                    value = 100
                    new_line = f"{split_line[0]:<15s}= {value}\n"
                    ofile.write(new_line)
                elif split_line[0] + f"_{filter}" in final_fit_dict:
                    param = split_line[0]
                    value = final_fit_dict[param + f"_{filter}"]
                    error = value/20 if param != "t0" else 30./86400
                    new_line = f"{param:<15s}= {value} {error} {error/10} 1 1\n"
                    ofile.write(new_line)
                else:
                    ofile.write(line)

    # Run lroche using the updated parameter_files. This must use the input data to obtain a proper scale factor
    cmd = f"singularity exec {lcurve_path} lroche parameters_{filter}_mcmc.txt object_{filter}_lcurve.txt 0 52723 1 roche_output_{filter}.txt none yes"
    output, error = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True).communicate()
    print(error)
    for line in output.split("\n"):
        if "scale factor" in line.lower():
            scale_factors[filter] = float(line.split()[-1])
            break
    print(f"================================================================================")
    print(cmd)
    print(output)

    # Run lroche again to create well-sampled model light curves, without gaps seen in the observed data.
    cmd = f"singularity exec {lcurve_path} lroche parameters_{filter}_mcmc.txt model_placeholder_{filter}.txt 0 52723 1 roche_output_{filter}_free.txt none no 1"
    print(cmd)
    output, error = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True).communicate()
    

with open("scale_factors_mcmc.csv", "w") as ofile:
    for filter in scale_factors:
        ofile.write(f"{filter},{scale_factors[filter]}\n")
#

In [ ]:
# Run LCURVE's "visualise" command to generate an image of the system at different orbital phases

image_path = "visualization_frames/" 
for k in os.listdir(image_path): # Remove all existing images first.
    os.remove(image_path + k)

# Define output plot parameters
xmin, xmax = -1, 1
ymin, ymax = -1, 1
n_images = 40
phase_min, phase_max = 0, 1

os.system(f"singularity exec {lcurve_path} visualise parameters_{filter}_mcmc.txt {n_images} {phase_min} {phase_max} /cps {xmin} {xmax} {ymin} {ymax} sdOB");
os.system("ps2pdf pgplot.ps"); os.remove("pgplot.ps"); # Use imagemagick to convert the postscript to a PDF
os.system("convert pgplot.pdf pgplot.jpg");            # Use imagemagick to convert the PDF into N .jpg images (pgplot-1.jpg, pgplot-2.jpg, . . .)
os.system("mv pgplot*jpg visualization_frames/")       # Organize these images into a different directory.
os.remove("pgplot.pdf")


fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(111)
ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)

original_image_filenames = [image_path+k for k in os.listdir(image_path) if "pgplot-" in k and "._" not in k and ".jpg"==k[-4:]][1:] # Skip the first image, which is usually zoomed in for some reason.

# Rename the jpg image files such that they use a numbering system with leading zeros (01, 02, 03, . . .) for sorting purposes.
image_name_lengths = [len(k) for k in original_image_filenames]
final_image_filenames = []
for filename in original_image_filenames:
    input_name = filename
    while len(filename) < max(image_name_lengths):
        filename = filename.replace("-", "-0")

    if input_name != filename:
        os.rename(input_name, filename)
    final_image_filenames.append(filename)

# Generate a list containing matplotlib objects, which will be the animation frames.
frames = []
for image_filename in sorted(final_image_filenames):
    img = mpimg.imread(image_filename)
    image = ax.imshow(img, animated=False)
    frames.append([image])
    
plt.tight_layout()
ani = animation.ArtistAnimation(fig=fig, artists=frames, interval=100)
plt.close(fig) # Prevent the blank figure box from showing after the animation. Call the "ani" variable after this.
ani


In [ ]:
# Build a corner plot for the manuscript, only including a few columns.

df2 = df[["q", "iangle", "t2", "M2", "M1", "R2vol"]]
g = sns.PairGrid(df2, corner=True, diag_sharey=False, despine=True, aspect=2)
g = g.map_diag(plt.hist, bins=25, histtype="step", color="black", density=True)
g = g.map_offdiag(plt.scatter, alpha=10./100, marker=".", color="black", s=1)


sigma2 = True
best_parameters = []
for i, k in enumerate(df2.keys()):
    fifty     = np.percentile(df2[k], 50)
    onesigup  = np.percentile(df2[k], 84.13) - fifty
    onesigdwn = fifty - np.percentile(df2[k], 15.87)
    
    if sigma2:
        twosigup = np.percentile(df2[k], 97.725) - fifty
        twosigdwn = fifty - np.percentile(df2[k], 2.275)
    
    print(f"{i:>2d} {k:>15s}   {fifty:>}  {onesigup:>}  {onesigdwn:>}")

    if k in header:
        best_parameters.append(fifty)
    
    title00 = f"{k} = ${fifty:>.5f}^{{+{onesigup:<.5f}}}_{{-{onesigdwn:<.5f}}}$"
            
    if k in ["t1", "t2", "temp_disc", "temp_spot", "temp_edge", "stsp11_tcen", "stsp12_tcen"]:
        if fifty > 1000:
            title00 = f"{k} = ${10*int(fifty/10):>5.0f}^{{+{10*int(onesigup/10):<5.0f}}}_{{-{10*int(onesigdwn/10):<5.0f}}}$"
        else:
            title00 = f"{k} = ${fifty:>6.2f}^{{+{onesigup:<6.2f}}}_{{-{onesigdwn:<6.2f}}}$"    

    elif "t0" in k:
        title00 = f"{k} = ${fifty:>5.6f}^{{+{onesigup:<.3e}}}_{{-{onesigdwn:<.3e}}}$"

    elif k in ["M2", "M1"]:
        title00 = f"{k} = ${fifty:>5.3f}^{{+{onesigup:<.3f}}}_{{-{onesigdwn:<.3f}}}$"

    elif k in ["r2", "r1", "R2vol", "R1vol", "R2", "R1"]:
        title00 = f"{k} = ${fifty:>5.4f}^{{+{onesigup:<.4f}}}_{{-{onesigdwn:<.4f}}}$"
        
    
    elif k in ["K2", "K1", "velocity_scale", "vsini"]:
        title00 = f"{k} = ${fifty:>5.1f}^{{+{onesigup:<.1f}}}_{{-{onesigdwn:<5.1f}}}$"
        
    else:
        title00 = f"{k} = ${fifty:>6.4f}^{{+{onesigup:>6.4f}}}_{{-{onesigdwn:>6.4f}}}$"

    g.axes[i,i].set_title(title00, loc="right")
    g.axes[i,i].axvline(x=fifty, color="crimson", linewidth=1)
    g.axes[i,i].axvline(x=fifty+onesigup, color="crimson", linestyle="--", linewidth=1)
    g.axes[i,i].axvline(x=fifty-onesigdwn, color="crimson", linestyle="--", linewidth=1)
    if sigma2:
        g.axes[i,i].axvline(x=fifty+twosigup, color="blue", linestyle="--", linewidth=1)
        g.axes[i,i].axvline(x=fifty-twosigdwn, color="blue", linestyle="--", linewidth=1)
print()
plt.show()
g.savefig("mini_corner_plot.jpg", dpi=150)

#